# 04 tables and figures

Design decisions: every manuscript number is derived from the committed result tables in `reports/<run>` by `src/sih_report.py`; raw data are read only for the dataset-composition table (manifests) and the Whelan receiver self-report table (live ULogs). Outputs go to `reports/<run>/tables` (CSV plus `tables.md`) and `figures/<run>` (PNG and PDF).

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
RUN, FEAT = "v4", "features_v3"
subprocess.run(["pip", "install", "-q", "pyulog", "pandas", "matplotlib"], check=True)
args = [sys.executable, str(P.SRC / "sih_report.py"), "--reports", str(P.REPORTS / RUN), "--figures", str(P.FIGURES / RUN),
        "--manifests", str(P.SIH_RUNS / "sih_flights_v2"), str(P.SIH_RUNS / "sih_flights_v2_nofix"),
        "--whelan_dir", str(P.FEATURES / FEAT / "whelan_live")]
r = subprocess.run(args, capture_output=True, text=True)
print(r.stdout[-12000:], r.stderr[-2000:])


In [ ]:
from IPython.display import Image, display
for name in ("f_leakage_audit", "f_risk_coverage", "f_conformal_stacked", "f_unseen_subtype_collapse",
             "f_whelan_timelines", "f_feature_importance", "f_flight_model_coefficients"):
    display(Image(filename=str(P.FIGURES / RUN / f"{name}.png"), width=760))


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, subprocess
os.chdir("/content/drive/MyDrive/UAV_GNSS_Research/uav-gnss-triage")
subprocess.run(["pip", "install", "-q", "nbconvert", "jupyter"], check=False)
def commit(msg):
    r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
    print(r.stdout, r.stderr)

In [ ]:
commit("v4: balanced test sets, abstention decomposition, flight-level calibration, paired leakage audit, aggregation ablations, hold-out controls, live-log events")